In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Define paths
BASE_PATH = Path('../steam_data_20251208')
REVIEWS_PATH = BASE_PATH / 'reviews'
METADATA_PATH = BASE_PATH / 'games_metadata_20251208.csv'
PLAYER_COUNT_PATH = BASE_PATH / 'player_count_history.csv'

## 1. Load and Explore Data

In [2]:
# Load games metadata
games_df = pd.read_csv(METADATA_PATH)
print(f"Total games in metadata: {len(games_df)}")
print(f"\nColumns: {games_df.columns.tolist()}")
print("\nGenres distribution:")
print(games_df['primary_genre_query'].value_counts())

Total games in metadata: 120

Columns: ['appid', 'primary_genre_query', 'name', 'type', 'release_date', 'is_free', 'developers', 'publishers', 'genres', 'short_description', 'num_reviews_in_query', 'review_score', 'review_score_desc', 'total_positive', 'total_negative', 'total_reviews']

Genres distribution:
primary_genre_query
sports                    10
horror                    10
science_fiction           10
exploration_open_world    10
anime                     10
survival                  10
action_fps                10
hidden_object             10
rpg_action                10
casual                    10
puzzle_matching           10
visual_novel              10
Name: count, dtype: int64


In [3]:
# Load player count history for popularity analysis
player_count_df = pd.read_csv(PLAYER_COUNT_PATH)
print(f"Player count records: {len(player_count_df)}")
print(f"\nColumns: {player_count_df.columns.tolist()}")
print("\nSample data:")
player_count_df.head()

Player count records: 34160

Columns: ['timestamp', 'appid', 'player_count']

Sample data:


,timestamp,appid,player_count
0,2025-12-10 23:20:00,1566200,66
1,2025-12-10 23:20:00,1324350,2
2,2025-12-10 23:20:00,2494350,201
3,2025-12-10 23:20:00,853200,1
4,2025-12-10 23:20:00,301120,136


In [4]:
# Calculate average player count per game for popularity tiers
popularity_df = player_count_df.groupby('appid').agg({
    'player_count': ['mean', 'max', 'std']
}).reset_index()
popularity_df.columns = ['appid', 'avg_player_count', 'max_player_count', 'std_player_count']
popularity_df = popularity_df.fillna(0)

# Create popularity tiers based on average player count
popularity_df['popularity_tier'] = pd.qcut(
    popularity_df['avg_player_count'].rank(method='first'), 
    q=3, 
    labels=['low', 'medium', 'high']
)

print("Popularity tier distribution:")
print(popularity_df['popularity_tier'].value_counts())
print("\nPopularity stats:")
popularity_df.groupby('popularity_tier')['avg_player_count'].describe()

Popularity tier distribution:
popularity_tier
low       41
high      41
medium    40
Name: count, dtype: int64

Popularity stats:


,count,mean,std,min,25%,50%,75%,max
popularity_tier,,,,,,,,
low,41.0,1.981446,1.704338,0.000000,0.410714,1.860714,2.885714,6.167857
medium,40.0,22.121339,13.152873,6.557143,10.895536,18.560714,34.329464,48.635714
high,41.0,3702.175523,15236.208798,52.835714,99.242857,164.067857,450.842857,91406.960714


In [5]:
# Merge popularity data with games metadata
games_df = games_df.merge(popularity_df[['appid', 'avg_player_count', 'popularity_tier']], 
                          on='appid', how='left')

# Fill missing popularity data
games_df['avg_player_count'] = games_df['avg_player_count'].fillna(0)
games_df['popularity_tier'] = games_df['popularity_tier'].fillna('low')

# Parse release date and create era tiers
def parse_release_year(date_str):
    """Extract year from release date string."""
    if pd.isna(date_str):
        return None
    try:
        # Handle formats like "17 Nov, 2008" or "Nov 17, 2008"
        match = re.search(r'(\d{4})', str(date_str))
        if match:
            return int(match.group(1))
    except:
        pass
    return None

games_df['release_year'] = games_df['release_date'].apply(parse_release_year)

# Create release era tiers
def get_release_era(year):
    if pd.isna(year):
        return 'unknown'
    if year < 2020:
        return 'classic'
    elif year < 2024:
        return 'recent'
    else:
        return 'new'

games_df['release_era'] = games_df['release_year'].apply(get_release_era)

print("Release era distribution:")
print(games_df['release_era'].value_counts())
print("\nFree vs Paid:")
print(games_df['is_free'].value_counts())

Release era distribution:
release_era
new        41
classic    40
recent     37
unknown     2
Name: count, dtype: int64

Free vs Paid:
is_free
False    111
True       7
Name: count, dtype: int64


In [6]:
# Load all reviews and create a master dataframe
def load_all_reviews(reviews_path):
    """Load all review CSV files from all genre folders."""
    all_reviews = []
    genre_folders = [f for f in reviews_path.iterdir() if f.is_dir()]
    
    print(f"Found {len(genre_folders)} genre folders")
    
    for genre_folder in genre_folders:
        genre = genre_folder.name
        csv_files = list(genre_folder.glob('reviews_*.csv'))
        
        for csv_file in csv_files:
            try:
                df = pd.read_csv(csv_file)
                df['genre'] = genre
                df['source_file'] = csv_file.name
                all_reviews.append(df)
            except Exception as e:
                print(f"Error loading {csv_file}: {e}")
    
    return pd.concat(all_reviews, ignore_index=True)

print("Loading all reviews...")
reviews_df = load_all_reviews(REVIEWS_PATH)
print(f"\nTotal reviews loaded: {len(reviews_df):,}")
print(f"Columns: {reviews_df.columns.tolist()}")

Loading all reviews...
Found 12 genre folders

Total reviews loaded: 103,946
Columns: ['appid', 'recommendationid', 'review_text', 'voted_up', 'timestamp_created', 'votes_up', 'votes_funny', 'weighted_vote_score', 'comment_count', 'playtime_at_review_minutes', 'playtime_forever_minutes', 'steamid', 'genre', 'source_file']


In [7]:
# Basic review statistics
print("REVIEW DATA SUMMARY")
print(f"\nTotal reviews: {len(reviews_df):,}")
print(f"Unique games: {reviews_df['appid'].nunique()}")
print(f"Unique reviewers: {reviews_df['steamid'].nunique():,}")

print("\nSentiment distribution (voted_up):")
sentiment_dist = reviews_df['voted_up'].value_counts()
print(f"  Positive (True): {sentiment_dist.get(True, 0):,} ({sentiment_dist.get(True, 0)/len(reviews_df)*100:.1f}%)")
print(f"  Negative (False): {sentiment_dist.get(False, 0):,} ({sentiment_dist.get(False, 0)/len(reviews_df)*100:.1f}%)")

print("\nReviews per genre:")
print(reviews_df['genre'].value_counts())

REVIEW DATA SUMMARY

Total reviews: 103,946
Unique games: 120
Unique reviewers: 99,382

Sentiment distribution (voted_up):
  Positive (True): 70,633 (68.0%)
  Negative (False): 33,313 (32.0%)

Reviews per genre:
genre
exploration_open_world    15705
action_fps                12459
science_fiction           12263
rpg_action                11482
casual                    11104
survival                   9498
puzzle_matching            8853
anime                      7243
horror                     6419
sports                     3806
hidden_object              2571
visual_novel               2543
Name: count, dtype: int64


## 2. Quality Filtering

In [8]:
# Add review text length column
reviews_df['text_length'] = reviews_df['review_text'].astype(str).apply(len)

print("Review length statistics (before filtering):")
print(reviews_df['text_length'].describe())

print("\nReviews by length category:")
print(f"  Very short (< 50 chars): {(reviews_df['text_length'] < 50).sum():,}")
print(f"  Short (50-150 chars): {((reviews_df['text_length'] >= 50) & (reviews_df['text_length'] < 150)).sum():,}")
print(f"  Medium (150-400 chars): {((reviews_df['text_length'] >= 150) & (reviews_df['text_length'] < 400)).sum():,}")
print(f"  Long (400+ chars): {(reviews_df['text_length'] >= 400).sum():,}")

Review length statistics (before filtering):
count    103946.000000
mean        355.804870
std         737.525013
min           1.000000
25%          31.000000
50%         107.000000
75%         344.000000
max       15991.000000
Name: text_length, dtype: float64

Reviews by length category:
  Very short (< 50 chars): 34,669
  Short (50-150 chars): 24,936
  Medium (150-400 chars): 21,471
  Long (400+ chars): 22,870


In [9]:
def is_likely_english(text):
    """
    Simple heuristic to detect if text is likely English.
    Uses character-based detection.
    """
    if pd.isna(text):
        return False
    
    text = str(text)
    
    # Count ASCII letters vs non-ASCII characters
    ascii_letters = sum(1 for c in text if c.isascii() and c.isalpha())
    non_ascii = sum(1 for c in text if not c.isascii())
    total_letters = ascii_letters + non_ascii
    
    if total_letters == 0:
        return False
    
    # If less than 70% ASCII letters, probably not English
    ascii_ratio = ascii_letters / total_letters
    if ascii_ratio < 0.70:
        return False
    
    # Check for common English words (simple validation)
    common_words = {'the', 'a', 'is', 'it', 'to', 'and', 'of', 'in', 'for', 'this', 
                    'that', 'but', 'not', 'you', 'game', 'play', 'good', 'bad', 
                    'like', 'really', 'very', 'fun', 'great', 'best', 'just', 'get'}
    words = set(text.lower().split())
    english_word_count = len(words.intersection(common_words))
    
    # If text has at least 2 common English words, likely English
    return english_word_count >= 2

def is_extreme_spam(text):
    """
    Only detect extreme spam cases - ASCII art and highly repetitive content.
    Relaxed to keep template-style reviews.
    """
    if pd.isna(text):
        return True
    
    text = str(text)
    
    # Only flag extreme ASCII art (very high counts)
    if text.count('/') > 20 or text.count('|') > 20 or text.count('\\') > 20:
        return True
    
    # Only flag extremely repetitive content (same word repeated many times)
    words = text.split()
    if len(words) > 10:
        unique_ratio = len(set(words)) / len(words)
        if unique_ratio < 0.15:
            return True
    
    return False

# Apply quality filters
print("Applying quality filters...")
original_count = len(reviews_df)

# Filter 1: Minimum length (50 characters)
reviews_df = reviews_df[reviews_df['text_length'] >= 50].copy()
print(f"After min length filter (50 chars): {len(reviews_df):,} ({len(reviews_df)/original_count*100:.1f}%)")

# Filter 2: English-only reviews
print("Detecting English reviews...")
reviews_df['is_english'] = reviews_df['review_text'].apply(is_likely_english)
english_count_before = len(reviews_df)
reviews_df = reviews_df[reviews_df['is_english']].copy()
print(f"After English filter: {len(reviews_df):,} ({len(reviews_df)/original_count*100:.1f}%)")
print(f"  (Removed {english_count_before - len(reviews_df):,} non-English reviews)")

# Filter 3: Remove only extreme spam (relaxed - keeps templates)
reviews_df['is_spam'] = reviews_df['review_text'].apply(is_extreme_spam)
spam_count = reviews_df['is_spam'].sum()
reviews_df = reviews_df[~reviews_df['is_spam']].copy()
print(f"After extreme spam filter: {len(reviews_df):,} ({len(reviews_df)/original_count*100:.1f}%)")
print(f"  (Removed {spam_count:,} extreme spam reviews)")

# Filter 4: Remove duplicates by recommendation ID
reviews_df = reviews_df.drop_duplicates(subset=['recommendationid']).copy()
print(f"After deduplication: {len(reviews_df):,} ({len(reviews_df)/original_count*100:.1f}%)")

print(f"\nTotal removed: {original_count - len(reviews_df):,} reviews")

Applying quality filters...
After min length filter (50 chars): 69,277 (66.6%)
Detecting English reviews...
After English filter: 66,317 (63.8%)
  (Removed 2,960 non-English reviews)
After extreme spam filter: 66,141 (63.6%)
  (Removed 176 extreme spam reviews)
After deduplication: 66,140 (63.6%)

Total removed: 37,806 reviews


In [10]:
# Sentiment distribution after filtering
print("Sentiment distribution after filtering:")
sentiment_dist = reviews_df['voted_up'].value_counts()
print(f"  Positive (True): {sentiment_dist.get(True, 0):,} ({sentiment_dist.get(True, 0)/len(reviews_df)*100:.1f}%)")
print(f"  Negative (False): {sentiment_dist.get(False, 0):,} ({sentiment_dist.get(False, 0)/len(reviews_df)*100:.1f}%)")

Sentiment distribution after filtering:
  Positive (True): 40,441 (61.1%)
  Negative (False): 25,699 (38.9%)


## 3. Create Stratification Buckets

In [11]:
# Create review length tiers (target: 20% short, 40% medium, 40% long)
def get_length_tier(length):
    if length < 150:
        return 'short'
    elif length < 400:
        return 'medium'
    else:
        return 'long'

reviews_df['length_tier'] = reviews_df['text_length'].apply(get_length_tier)

print("Length tier distribution:")
print(reviews_df['length_tier'].value_counts())
print("\nPercentages:")
print(reviews_df['length_tier'].value_counts(normalize=True) * 100)

Length tier distribution:
length_tier
short     22655
long      22383
medium    21102
Name: count, dtype: int64

Percentages:
length_tier
short     34.253099
long      33.841851
medium    31.905050
Name: proportion, dtype: float64


In [12]:
# Create playtime tiers (<2h, 2-10h, 10h+)
def get_playtime_tier(minutes):
    if pd.isna(minutes):
        return 'unknown'
    hours = minutes / 60
    if hours < 2:
        return 'short'
    elif hours < 10:
        return 'medium'
    else:
        return 'long'

reviews_df['playtime_tier'] = reviews_df['playtime_at_review_minutes'].apply(get_playtime_tier)

print("Playtime tier distribution:")
print(reviews_df['playtime_tier'].value_counts())
print("\nPercentages:")
print(reviews_df['playtime_tier'].value_counts(normalize=True) * 100)

Playtime tier distribution:
playtime_tier
long       30102
medium     23943
short      12040
unknown       55
Name: count, dtype: int64

Percentages:
playtime_tier
long       45.512549
medium     36.200484
short      18.203810
unknown     0.083157
Name: proportion, dtype: float64


In [13]:
# Merge game metadata with reviews
reviews_df = reviews_df.merge(
    games_df[['appid', 'name', 'primary_genre_query', 'is_free', 'popularity_tier', 'release_era']],
    on='appid',
    how='left'
)

# Use primary_genre_query as the main genre (from metadata)
# Fall back to the folder-based genre if metadata is missing
reviews_df['primary_genre'] = reviews_df['primary_genre_query'].fillna(reviews_df['genre'])

print("Reviews with metadata merged:")
print(f"Total reviews: {len(reviews_df):,}")
print("\nPrimary genre distribution:")
print(reviews_df['primary_genre'].value_counts())

Reviews with metadata merged:
Total reviews: 66,140

Primary genre distribution:
primary_genre
science_fiction           9043
exploration_open_world    8799
action_fps                7706
rpg_action                7522
casual                    6957
survival                  6592
puzzle_matching           5678
anime                     4631
horror                    3737
sports                    2498
hidden_object             1501
visual_novel              1476
Name: count, dtype: int64


## 4. Analyze Available Reviews Per Game

In [14]:
# Analyze positive/negative review availability per game
game_review_stats = reviews_df.groupby('appid').agg({
    'voted_up': ['sum', 'count'],
    'name': 'first',
    'primary_genre': 'first',
    'popularity_tier': 'first',
    'is_free': 'first',
    'release_era': 'first'
}).reset_index()

game_review_stats.columns = ['appid', 'positive_count', 'total_count', 'name', 
                              'primary_genre', 'popularity_tier', 'is_free', 'release_era']
game_review_stats['negative_count'] = game_review_stats['total_count'] - game_review_stats['positive_count']
game_review_stats['positive_ratio'] = game_review_stats['positive_count'] / game_review_stats['total_count']

# Calculate maximum balanced samples possible per game
game_review_stats['max_balanced_per_class'] = game_review_stats[['positive_count', 'negative_count']].min(axis=1)

print("Game review statistics:")
print(game_review_stats[['name', 'total_count', 'positive_count', 'negative_count', 'positive_ratio', 'max_balanced_per_class']].head(20))

Game review statistics:
                                                 name  total_count  \
0                                         Left 4 Dead          895   
1                                           BioShock™         1206   
2                                    Star Trek Online         1397   
3                                        World of Goo          736   
4   The Elder Scrolls IV: Oblivion® Game of the Ye...         1159   
5                                          Samorost 2          718   
6      Batman: Arkham City - Game of the Year Edition         1102   
7                                         To the Moon         1391   
8                                              Lucius         1180   
9   THE KING OF FIGHTERS '98 ULTIMATE MATCH FINAL ...          511   
10                           Far Cry 3 - Blood Dragon         1317   
11                                     Battle Nations          687   
12        RollerCoaster Tycoon® 2: Triple Thrill Pack          733

In [15]:
# Identify games with very few negative reviews
print("Games with limited negative reviews (< 100):")
limited_negatives = game_review_stats[game_review_stats['negative_count'] < 100].sort_values('negative_count')
print(limited_negatives[['name', 'primary_genre', 'positive_count', 'negative_count', 'positive_ratio']])

print(f"\n{len(limited_negatives)} games have < 100 negative reviews")
print(f"\nTotal available balanced reviews: {game_review_stats['max_balanced_per_class'].sum() * 2:,}")

Games with limited negative reviews (< 100):
                                       name           primary_genre  \
53                TrymenT ―献给渴望改变的你― AlphA篇            visual_novel   
95                      Tenioha! feat. Mami            visual_novel   
100                            Folie Fatale            visual_novel   
98   Climb Challenge - Find Items Cyberpunk                  sports   
94                          Kuroinu 2 Redux            visual_novel   
..                                      ...                     ...   
61                          Winter Survival  exploration_open_world   
65                 The Jackbox Party Pack 8                  casual   
22                                 Overload              action_fps   
58                        Turbo Golf Racing                  sports   
39                         Prison Simulator              action_fps   

     positive_count  negative_count  positive_ratio  
53                9               0        1.000

## 5. Select 50 Games via Stratified Sampling

In [16]:
# Filter games with at least 50 reviews per class
MIN_REVIEWS_PER_CLASS = 50
eligible_games = game_review_stats[game_review_stats['max_balanced_per_class'] >= MIN_REVIEWS_PER_CLASS].copy()

print(f"Games eligible for selection (>= {MIN_REVIEWS_PER_CLASS} reviews per class): {len(eligible_games)}")
print("\nEligible games by genre:")
print(eligible_games['primary_genre'].value_counts())

Games eligible for selection (>= 50 reviews per class): 64

Eligible games by genre:
primary_genre
exploration_open_world    10
action_fps                 8
rpg_action                 7
casual                     7
science_fiction            7
puzzle_matching            6
survival                   6
sports                     4
anime                      4
horror                     2
hidden_object              2
visual_novel               1
Name: count, dtype: int64


In [17]:
def stratified_game_selection(games_df, n_games=50, seed=42):
    """
    Select n_games using stratified sampling across:
    - Genre (primary factor)
    - Popularity tier
    - Free/Paid status
    - Release era
    
    Uses a two-pass approach:
    1. First pass: allocate games proportionally to available eligible games per genre
    2. Prioritize genres with more available data
    """
    np.random.seed(seed)
    
    genres = games_df['primary_genre'].unique()
    n_genres = len(genres)
    
    # Calculate available games per genre
    genre_counts = games_df['primary_genre'].value_counts()
    total_eligible = len(games_df)
    
    print(f"Selecting up to {n_games} games from {n_genres} genres")
    print(f"Total eligible games: {total_eligible}")
    
    # Allocate games proportionally, with minimum of 1 per genre and maximum of available
    genre_allocations = {}
    remaining_slots = n_games
    
    # First, give each genre at least 1-2 games (if available)
    for genre in genres:
        available = genre_counts.get(genre, 0)
        min_alloc = min(2, available)  # At least 2 or all available
        genre_allocations[genre] = min_alloc
        remaining_slots -= min_alloc
    
    # Distribute remaining slots proportionally to genres with more data
    if remaining_slots > 0:
        # Sort genres by available games (descending) and max_balanced_per_class
        genre_priority = games_df.groupby('primary_genre')['max_balanced_per_class'].sum().sort_values(ascending=False)
        
        for genre in genre_priority.index:
            if remaining_slots <= 0:
                break
            available = genre_counts.get(genre, 0)
            current_alloc = genre_allocations[genre]
            can_add = min(remaining_slots, available - current_alloc, 3)  # Add up to 3 more
            if can_add > 0:
                genre_allocations[genre] += can_add
                remaining_slots -= can_add
    
    print("\nAllocation per genre:")
    for genre, alloc in sorted(genre_allocations.items(), key=lambda x: -x[1]):
        print(f"  {genre}: {alloc} games")
    
    # Select games from each genre
    selected_games = []
    for genre, n_select in genre_allocations.items():
        if n_select == 0:
            continue
            
        genre_games = games_df[games_df['primary_genre'] == genre].copy()
        
        # Score games: prioritize those with more balanced reviews
        genre_games['selection_score'] = (
            genre_games['max_balanced_per_class'] * 1.0 +  # Prefer balanced games
            np.random.rand(len(genre_games)) * 50  # Random component for diversity
        )
        
        # Try to get mix of popularity tiers
        selected_list = []
        for tier in ['high', 'medium', 'low']:  # Prefer high popularity first
            tier_games = genre_games[genre_games['popularity_tier'] == tier]
            if len(tier_games) > 0 and len(selected_list) < n_select:
                n_from_tier = max(1, (n_select - len(selected_list)) // 2)
                tier_sample = tier_games.nlargest(min(n_from_tier, len(tier_games)), 'selection_score')
                for _, row in tier_sample.iterrows():
                    if len(selected_list) < n_select:
                        selected_list.append(row)
        
        # If we still need more, add from remaining
        if len(selected_list) < n_select:
            already_selected = [g['appid'] for g in selected_list] if selected_list else []
            remaining = genre_games[~genre_games['appid'].isin(already_selected)]
            additional = remaining.nlargest(n_select - len(selected_list), 'selection_score')
            for _, row in additional.iterrows():
                selected_list.append(row)
        
        if selected_list:
            selected_games.extend(selected_list[:n_select])
    
    result = pd.DataFrame(selected_games)
    print(f"\nTotal selected games: {len(result)}")
    return result

# Select games (target 50, may get fewer if data is limited)
selected_games = stratified_game_selection(eligible_games, n_games=50, seed=RANDOM_SEED)

Selecting up to 50 games from 12 genres
Total eligible games: 64

Allocation per genre:
  action_fps: 5 games
  exploration_open_world: 5 games
  puzzle_matching: 5 games
  rpg_action: 5 games
  casual: 5 games
  science_fiction: 5 games
  survival: 5 games
  sports: 4 games
  anime: 4 games
  horror: 2 games
  hidden_object: 2 games
  visual_novel: 1 games

Total selected games: 48


In [18]:
# Display selected games summary
print("SELECTED GAMES SUMMARY")

print("\nBy Genre:")
print(selected_games['primary_genre'].value_counts())

print("\nBy Popularity Tier:")
print(selected_games['popularity_tier'].value_counts())

print("\nBy Release Era:")
print(selected_games['release_era'].value_counts())

print("\nBy Free/Paid:")
print(selected_games['is_free'].value_counts())

print(f"\nTotal balanced reviews available: {selected_games['max_balanced_per_class'].sum() * 2:,}")

SELECTED GAMES SUMMARY

By Genre:
primary_genre
action_fps                5
exploration_open_world    5
puzzle_matching           5
rpg_action                5
casual                    5
science_fiction           5
survival                  5
sports                    4
anime                     4
horror                    2
hidden_object             2
visual_novel              1
Name: count, dtype: int64

By Popularity Tier:
popularity_tier
high      26
medium    14
low        8
Name: count, dtype: int64

By Release Era:
release_era
classic    22
recent     15
new        10
unknown     1
Name: count, dtype: int64

By Free/Paid:
is_free
False    42
True      5
Name: count, dtype: int64

Total balanced reviews available: 31,622


In [19]:
# Display full list of selected games
print("\nSelected Games List:")
display_cols = ['appid', 'name', 'primary_genre', 'popularity_tier', 'is_free', 
                'positive_count', 'negative_count', 'max_balanced_per_class']
selected_games[display_cols].sort_values('primary_genre')


Selected Games List:


,appid,name,primary_genre,popularity_tier,is_free,positive_count,negative_count,max_balanced_per_class
43,973580,Sniper Ghost Warrior Contracts,action_fps,high,False,497,810,497
0,500,Left 4 Dead,action_fps,high,False,337,558,337
1,7670,BioShock™,action_fps,medium,False,460,746,460
28,541200,GTTOD: Get To The Orange Door,action_fps,low,False,510,278,278
34,751630,After the Fall®,action_fps,medium,False,673,359,359
30,589530,Hakuoki: Kyoto Winds,anime,medium,False,453,53,53
47,1105510,Yakuza 5 Remastered,anime,high,False,486,242,242
78,1858630,SWORD ART ONLINE Fractured Daydream,anime,high,False,422,254,254
59,1353230,Bomb Rush Cyberfunk,anime,high,False,502,215,215
11,251670,Battle Nations,casual,high,True,457,230,230


## 6. Sample Reviews with Flexible Per-Game Quotas

In [20]:
# Configuration
TARGET_TOTAL_REVIEWS = 50000
TARGET_PER_CLASS = TARGET_TOTAL_REVIEWS // 2  # 25,000 positive, 25,000 negative

# Target distribution for length tiers (within each sentiment class)
LENGTH_TIER_TARGETS = {'short': 0.20, 'medium': 0.40, 'long': 0.40}

# Target distribution for playtime tiers
PLAYTIME_TIER_TARGETS = {'short': 0.33, 'medium': 0.33, 'long': 0.34}

print(f"Target: {TARGET_TOTAL_REVIEWS:,} total reviews")
print(f"  - {TARGET_PER_CLASS:,} positive")
print(f"  - {TARGET_PER_CLASS:,} negative")

Target: 50,000 total reviews
  - 25,000 positive
  - 25,000 negative


In [21]:
def stratified_review_sampling(reviews_df, games_df, target_total=50000, seed=42):
    """
    Sample reviews with STRICT 50/50 sentiment balance and stratification by:
    - Length tier (20% short, 40% medium, 40% long) 
    - Playtime tier (33% each: light, moderate, heavy)
    
    The sampling is constrained by the limiting sentiment class (usually negative).
    """
    np.random.seed(seed)
    
    # Filter reviews to only include selected games
    selected_appids = set(games_df['appid'].unique())
    reviews = reviews_df[reviews_df['appid'].isin(selected_appids)].copy()
    
    print(f"Reviews from selected games: {len(reviews):,}")
    
    # Find max possible balanced reviews
    positive_reviews = reviews[reviews['voted_up']]
    negative_reviews = reviews[~reviews['voted_up']]
    
    max_per_class = min(len(positive_reviews), len(negative_reviews))
    target_per_class = min(target_total // 2, max_per_class)
    
    print(f"Available positive reviews: {len(positive_reviews)}")
    print(f"Available negative reviews: {len(negative_reviews)}")
    print(f"Target per sentiment class: {target_per_class}")
    print(f"Total target reviews: {target_per_class * 2}")
    
    # Define stratification targets
    length_targets = {'short': 0.20, 'medium': 0.40, 'long': 0.40}
    playtime_targets = {'short': 0.333, 'medium': 0.333, 'long': 0.334}
    
    def sample_sentiment_class(df, n_samples, class_name):
        """Sample from a single sentiment class with length/playtime stratification."""
        samples = []
        
        # Calculate targets for each length tier
        for length_tier, length_pct in length_targets.items():
            length_group = df[df['length_tier'] == length_tier]
            n_length_target = int(n_samples * length_pct)
            
            if len(length_group) == 0:
                continue
            
            # Within each length tier, try to balance playtime
            for playtime_tier, playtime_pct in playtime_targets.items():
                subset = length_group[length_group['playtime_tier'] == playtime_tier]
                n_target = max(1, int(n_length_target * playtime_pct))
                
                if len(subset) > 0:
                    n_sample = min(n_target, len(subset))
                    sampled = subset.sample(n=n_sample, random_state=seed)
                    samples.append(sampled)
        
        if not samples:
            # Fallback: random sample if stratification fails
            return df.sample(n=min(n_samples, len(df)), random_state=seed)
        
        sampled_df = pd.concat(samples, ignore_index=True)
        
        # If we need more, random sample from remaining
        if len(sampled_df) < n_samples:
            already_sampled = set(sampled_df['recommendationid'].values)
            remaining = df[~df['recommendationid'].isin(already_sampled)]
            n_more = n_samples - len(sampled_df)
            if len(remaining) > 0:
                additional = remaining.sample(n=min(n_more, len(remaining)), random_state=seed+1)
                sampled_df = pd.concat([sampled_df, additional], ignore_index=True)
        elif len(sampled_df) > n_samples:
            # If we have too many, randomly subsample
            sampled_df = sampled_df.sample(n=n_samples, random_state=seed)
        
        return sampled_df
    
    # Sample each sentiment class
    sampled_positive = sample_sentiment_class(positive_reviews, target_per_class, 'positive')
    sampled_negative = sample_sentiment_class(negative_reviews, target_per_class, 'negative')
    
    # Enforce exact balance
    final_size = min(len(sampled_positive), len(sampled_negative))
    sampled_positive = sampled_positive.head(final_size)
    sampled_negative = sampled_negative.head(final_size)
    
    print(f"\nFinal sampled positive: {len(sampled_positive)}")
    print(f"Final sampled negative: {len(sampled_negative)}")
    
    # Combine
    final_sample = pd.concat([sampled_positive, sampled_negative], ignore_index=True)
    final_sample = final_sample.sample(frac=1, random_state=seed).reset_index(drop=True)  # Shuffle
    
    return final_sample

# Perform stratified sampling
sampled_reviews = stratified_review_sampling(
    reviews_df, 
    selected_games, 
    target_total=50000, 
    seed=RANDOM_SEED
)

Reviews from selected games: 44,994
Available positive reviews: 24705
Available negative reviews: 20289
Target per sentiment class: 20289
Total target reviews: 40578

Final sampled positive: 20289
Final sampled negative: 20289


In [22]:
# Verify sampling distribution
print("SAMPLING DISTRIBUTION VERIFICATION")

print("\n1. Sentiment Balance:")
sentiment_counts = sampled_reviews['voted_up'].value_counts()
print(f"   Positive: {sentiment_counts.get(True, 0):,} ({sentiment_counts.get(True, 0)/len(sampled_reviews)*100:.1f}%)")
print(f"   Negative: {sentiment_counts.get(False, 0):,} ({sentiment_counts.get(False, 0)/len(sampled_reviews)*100:.1f}%)")

print("\n2. Length Tier Distribution:")
length_counts = sampled_reviews['length_tier'].value_counts(normalize=True) * 100
for tier in ['short', 'medium', 'long']:
    actual = length_counts.get(tier, 0)
    target = LENGTH_TIER_TARGETS.get(tier, 0) * 100
    print(f"   {tier}: {actual:.1f}% (target: {target:.0f}%)")

print("\n3. Playtime Tier Distribution:")
playtime_counts = sampled_reviews['playtime_tier'].value_counts(normalize=True) * 100
for tier in ['short', 'medium', 'long']:
    actual = playtime_counts.get(tier, 0)
    target = PLAYTIME_TIER_TARGETS.get(tier, 0) * 100
    print(f"   {tier}: {actual:.1f}% (target: {target:.0f}%)")

print("\n4. Genre Distribution:")
print(sampled_reviews['primary_genre'].value_counts())

print(f"\n5. Games Represented: {sampled_reviews['appid'].nunique()}")

SAMPLING DISTRIBUTION VERIFICATION

1. Sentiment Balance:
   Positive: 20,289 (50.0%)
   Negative: 20,289 (50.0%)

2. Length Tier Distribution:
   short: 30.2% (target: 20%)
   medium: 33.3% (target: 40%)
   long: 36.5% (target: 40%)

3. Playtime Tier Distribution:
   short: 19.3% (target: 33%)
   medium: 35.6% (target: 33%)
   long: 45.0% (target: 34%)

4. Genre Distribution:
primary_genre
science_fiction           6412
exploration_open_world    5016
action_fps                4816
rpg_action                4575
survival                  4535
casual                    4184
puzzle_matching           3676
anime                     2252
sports                    1937
horror                    1908
hidden_object              744
visual_novel               523
Name: count, dtype: int64

5. Games Represented: 48


## 7. Text Preprocessing

In [23]:
import sys
sys.path.insert(0, '..')
from preprocessing import preprocess_for_fasttext

# Apply preprocessing
print("Preprocessing text...")
sampled_reviews['processed_text'] = sampled_reviews['review_text'].apply(preprocess_for_fasttext)

# Show examples
print("\nPreprocessing examples:")
for i in range(3):
    print(f"\n--- Example {i+1} ---")
    print(f"Original: {sampled_reviews.iloc[i]['review_text'][:200]}...")
    print(f"Processed: {sampled_reviews.iloc[i]['processed_text'][:200]}...")

Preprocessing text...

Preprocessing examples:

--- Example 1 ---
Original: I read the comments, that it is different from JC3. And decided to buy it regardless. At the end, yes it is different, but still I had a lot of fun, passing it, they just made a bigger map and decided...
Processed: i read the comments , that it is different from jc3 . and decided to buy it regardless . at the end , yes it is different , but still i had a lot of fun , passing it , they just made a bigger map and ...

--- Example 2 ---
Original: Can't recommend this game in it's current state, only one server seems to have people playing on it and most of the people I've seen on it are toxic and unfriendly unless you're part of their friend g...
Processed: can ' t recommend this game in it ' s current state , only one server seems to have people playing on it and most of the people i ' ve seen on it are toxic and unfriendly unless you ' re part of their...

--- Example 3 ---
Original: I have no words for this mas

In [24]:
# Remove reviews that became too short after preprocessing
sampled_reviews['processed_length'] = sampled_reviews['processed_text'].apply(len)

before_count = len(sampled_reviews)
sampled_reviews = sampled_reviews[sampled_reviews['processed_length'] >= 30].copy()
after_count = len(sampled_reviews)

print(f"Removed {before_count - after_count} reviews with processed text < 30 chars")
print(f"Final dataset size: {len(sampled_reviews):,}")

Removed 3 reviews with processed text < 30 chars
Final dataset size: 40,575


## 8. Create Train/Validation/Test Splits

In [26]:
from sklearn.model_selection import train_test_split

# Create stratified split: 70% train, 15% validation, 15% test
# Stratify by sentiment and genre

# Create stratification key
sampled_reviews['stratify_key'] = (
    sampled_reviews['voted_up'].astype(str) + '_' + 
    sampled_reviews['primary_genre'].astype(str)
)

# First split: 70% train, 30% temp
train_df, temp_df = train_test_split(
    sampled_reviews,
    test_size=0.30,
    stratify=sampled_reviews['stratify_key'],
    random_state=RANDOM_SEED
)

# Second split: 50% of temp = 15% validation, 50% of temp = 15% test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df['stratify_key'],
    random_state=RANDOM_SEED
)

print("Dataset Splits:")
print(f"  Train: {len(train_df):,} ({len(train_df)/len(sampled_reviews)*100:.1f}%)")
print(f"  Validation: {len(val_df):,} ({len(val_df)/len(sampled_reviews)*100:.1f}%)")
print(f"  Test: {len(test_df):,} ({len(test_df)/len(sampled_reviews)*100:.1f}%)")

Dataset Splits:
  Train: 28,402 (70.0%)
  Validation: 6,086 (15.0%)
  Test: 6,087 (15.0%)


In [27]:
# Verify splits maintain balance
print("Sentiment balance verification:")
for name, df in [('Train', train_df), ('Validation', val_df), ('Test', test_df)]:
    pos_pct = df['voted_up'].mean() * 100
    print(f"  {name}: {pos_pct:.1f}% positive, {100-pos_pct:.1f}% negative")

Sentiment balance verification:
  Train: 50.0% positive, 50.0% negative
  Validation: 50.0% positive, 50.0% negative
  Test: 50.0% positive, 50.0% negative


## 9. Save Processed Datasets

In [28]:
# Create output directory
OUTPUT_DIR = Path('../processed_data')
OUTPUT_DIR.mkdir(exist_ok=True)

# Select columns to save
SAVE_COLUMNS = [
    'recommendationid',
    'appid',
    'review_text',
    'processed_text',
    'voted_up',
    'playtime_at_review_minutes',
    'playtime_tier',
    'text_length',
    'length_tier',
    'primary_genre',
    'name'
]

# Save datasets
train_df[SAVE_COLUMNS].to_csv(OUTPUT_DIR / 'train.csv', index=False)
val_df[SAVE_COLUMNS].to_csv(OUTPUT_DIR / 'validation.csv', index=False)
test_df[SAVE_COLUMNS].to_csv(OUTPUT_DIR / 'test.csv', index=False)

# Save full dataset as well
sampled_reviews[SAVE_COLUMNS].to_csv(OUTPUT_DIR / 'full_dataset.csv', index=False)

print(f"Datasets saved to {OUTPUT_DIR.absolute()}")
print(f"  - train.csv: {len(train_df):,} reviews")
print(f"  - validation.csv: {len(val_df):,} reviews")
print(f"  - test.csv: {len(test_df):,} reviews")
print(f"  - full_dataset.csv: {len(sampled_reviews):,} reviews")

Datasets saved to d:\Study\Code\Projects\web-mining-20251-steam-sentiment-analysis\notebooks\..\processed_data
  - train.csv: 28,402 reviews
  - validation.csv: 6,086 reviews
  - test.csv: 6,087 reviews
  - full_dataset.csv: 40,575 reviews


In [29]:
# Save selected games metadata
selected_games.to_csv(OUTPUT_DIR / 'selected_games.csv', index=False)
print(f"  - selected_games.csv: {len(selected_games)} games")

# Save preprocessing summary
summary = {
    'total_reviews': int(len(sampled_reviews)),
    'train_size': int(len(train_df)),
    'val_size': int(len(val_df)),
    'test_size': int(len(test_df)),
    'num_games': int(sampled_reviews['appid'].nunique()),
    'num_genres': int(sampled_reviews['primary_genre'].nunique()),
    'positive_count': int(sampled_reviews['voted_up'].sum()),
    'negative_count': int((~sampled_reviews['voted_up']).sum()),
    'positive_ratio': float(sampled_reviews['voted_up'].mean()),
    'random_seed': RANDOM_SEED,
    'min_text_length': 50,
    'length_tier_targets': LENGTH_TIER_TARGETS,
    'playtime_tier_targets': PLAYTIME_TIER_TARGETS
}

import json
with open(OUTPUT_DIR / 'preprocessing_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("  - preprocessing_summary.json")
print("\nPREPROCESSING COMPLETE!")

print("\n Final Dataset Statistics:")
print(f"  Total reviews: {len(sampled_reviews):,}")
print(f"  Positive: {sampled_reviews['voted_up'].sum():,} ({sampled_reviews['voted_up'].mean()*100:.1f}%)")
print(f"  Negative: {(~sampled_reviews['voted_up']).sum():,} ({(~sampled_reviews['voted_up']).mean()*100:.1f}%)")
print(f"  Games: {sampled_reviews['appid'].nunique()}")
print(f"  Genres: {sampled_reviews['primary_genre'].nunique()}")

  - selected_games.csv: 48 games
  - preprocessing_summary.json

PREPROCESSING COMPLETE!

 Final Dataset Statistics:
  Total reviews: 40,575
  Positive: 20,289 (50.0%)
  Negative: 20,286 (50.0%)
  Games: 48
  Genres: 12
